# Dhara — contrastive fine-tuning of BGE-m3 (T4 / LoRA)

This is the experiment the whole project exists to run: does domain fine-tuning
close the lexical gap between colloquial citizen Bangla and formal legal Bangla?

**The comparison is fine-tuned BGE-m3 vs zero-shot BGE-m3**, not neural vs
lexical. The zero-shot control was locked on 2026-08-31 before any training:
provision-level R@10 = 0.324 overall, 0.222 English-gold, 0.546 Bengali-gold,
over 552 human-adjudicated questions and all 39,484 chunks
(`results/runs/bge_m3_zeroshot.json`).

### Attempt 4 = attempt 3's training + the Act-title document template

Attempt 3 worked. Against the locked zero-shot control, over 552 questions and
all 39,484 chunks: R@10 0.324 -> 0.350, R@20 0.413 -> 0.437, R@50 0.534 -> 0.560,
all significant under a paired bootstrap, and largest on the English-gold slice
(R@10 0.222 -> 0.249) which is the cross-language half the project exists to
measure. Canaries clean: corpus cosine 0.988, query concentration +0.011.

Separately, putting the Act title into the indexed document -- no training at all
-- gave R@1 0.096 -> 0.132 and R@10 0.324 -> 0.346.

Those two have never been combined, because attempt 3 deliberately kept the
control's document template so the comparison isolated the weights. This run
stacks them: same training recipe, Act-title template applied to both the
training texts and the index. It also saves the LoRA adapter, which attempt 3
did not.

### This is attempt 3. What the first two taught us

**Attempt 1 — embedding collapse.** R@10 fell 0.324 → 0.060. Two bugs: negative
mining excluded a candidate only if it was relevant to *that* topic, so 7.7% of
mined negatives were another pair's actual correct answer; and each pair was
exploded into 4 duplicate rows at batch 8, making same-batch collisions routine.
Both fixed (`scripts/17_mine_negatives.py` global exclusion; one `InputExample`
per pair).

**Attempt 2 — query-space hubness.** No collapse (corpus cosine 0.83, healthy)
but recall went flat and English R@1/R@5 got significantly *worse*. Cause: the
queries bunched together (mean cosine to the query centroid 0.699 → 0.822) and
whatever documents sat near that direction became universal top-1 attractors —
one chunk went from 4 to 43 first-place slots. Not memorisation: that chunk
appeared zero times as a training positive. See `scripts/20_hubness_audit.py`.

### What changed for attempt 3

1. **Training questions are now narratives, not one-liners.** Measuring the
   training set against the real one found the collapsed axis: real questions
   are newspaper legal-advice letters with median 74 words and sd 56.3; the
   templates were one-liners at median 11 words, sd 3.41. The generator now
   composes `opener + background clauses + core ask + closing`
   (`configs/synth_narrative.yaml`), giving median 55 words and sd 30.3 — the
   distributions substantially overlap now instead of being disjoint.
   (Current figures after the opener fix: median 57, sd 30.3.)
2. **Wrong labels dropped.** Attempts 1 and 2 trained on two kinds of label:
   `anchor` positives named by hand in `configs/synth_anchors.yaml`, and
   `topic_signature` positives found by keyword matching. Ten random signature
   pairs were inspected on 2026-09-04 and roughly eight were wrong — a
   খোরপোশ (wife's maintenance) question labelled with *The Chartered Accountants
   Ordinance, 1961*, "Maintenance of branch offices"; a land-dispossession
   question labelled with the Penal Code's "Possession of coin by person who
   knew it to be counterfeit"; দেনমোহর labelled with the Rangamati Hill District
   Council Act. English keyword collisions, every one. Under MNRL a wrong
   positive does not merely fail to teach, it pulls an unrelated provision
   toward citizen phrasing. This notebook now trains on the `anchor` variant
   only: 4,408 pairs over 168 hand-verified positive provisions, down from
   7,586 over 1,557.

   A second labelling defect was found the same day and fixed in the generator:
   positives were selected per *topic*, so each question carried a median of 6
   "correct" provisions of which typically one or two answered the actual ask.
   `select_positives()` now matches the question's intent against the provision
   title (median 3 positives, 53% intent-matched, the rest falling back to the
   full anchor list and counted).
3. **In-batch contradictions removed.** Each question carries a median of 3
   anchored positives, so one row per pair still puts the same question in a
   batch with itself, and two rows often share a positive. MNRL reads both as
   "your correct answer is a negative". Measured on this dataset at batch 16
   with shuffled batching: 65 same-anchor and 190 same-positive collisions
   across one epoch's 333 batches. Batching is now
   `BatchSamplers.NO_DUPLICATES` via the Trainer API, which takes both to zero;
   the notebook prints the before and after.
4. **Training questions are narratives, not one-liners** (see above), and the
   generator's third collapsed axis — opening bigrams — was fixed too.
5. **Less drift**: 1 epoch instead of 2, LR 1e-5 instead of 2e-5.
6. **A concentration guard** in the canary cell, which attempt 2 would have
   tripped before its corpus embed rather than after its full eval.

The trade accepted in change 2 is coverage: 173 distinct positive provisions
instead of 1,557. That is deliberate. Transfer here has to come from learning
the register mapping — colloquial narrative to formal legal text — and the
strict gold-provision exclusion means the model never trains on a provision it
will be tested on either way. A correct mapping over 173 provisions teaches
that better than a mapping over 1,557 where a third of the targets are the
wrong section of the wrong Act.

Worth stating plainly: measured against the *base* encoder, attempt 2's short
templates were no more concentrated than the real questions (0.6983 vs 0.6990).
So the length mismatch was a genuine train/test distribution gap, but it was not
proven to be the cause of the concentration — that came from the training
dynamics, which is what changes 3 addresses. Two fixes for two distinct
problems, and only the second one directly targets the observed failure.

### What this notebook must not do

- It must not touch `gold_test_v1.jsonl`. Training data is synthetic and
  topic-generated; the 552 probe questions are for scoring only, at the end.
- It must not re-tune anything against the probe set. One run, one number.
- It must emit the index in exactly the layout `scripts/13_eval_dense_index.py`
  reads, so the fine-tuned and zero-shot numbers come out of the same code path.
  Two different scoring scripts is how a fake improvement gets published.

### Upload these four files

| file | what it is |
|---|---|
| `corpus_v1.jsonl.gz` | the frozen corpus (12 MB gzipped; the plain 138 MB file truncates in Colab's upload widget) |
| `train_anchor_v1_negatives.jsonl` | 4,408 pairs / 1,436 distinct questions over 168 hand-anchored positive provisions, 8 mined hard negatives each, no gold provision among the positives |
| `dev_anchor_v1.jsonl` | 748 pairs across 5 held-out topics |
| `probe_questions.jsonl` | the 552 evaluation questions |

The `anchor` variant is two filters at once. It drops every pair whose positive
is a gold provision (as `strict` does — 42% of gold provisions also appear as
synthetic positives, so without this a gain could be memorisation rather than
transfer), *and* it drops the keyword-matched labels, keeping only positives
named by hand in `configs/synth_anchors.yaml`.

Run `python scripts/23_preflight.py --variant anchor` before uploading. It
checks gold isolation, negatives/split alignment, label purity, and prints SHA-1
fingerprints to compare against what actually lands here. Expected on the
current files:

```
58ac831dc3b6  train_anchor_v1.jsonl             9.8 MB
d8d0e7cf7f71  train_anchor_v1_negatives.jsonl  10.9 MB
e39d04911133  dev_anchor_v1.jsonl               1.7 MB
c810e7d42787  probe_questions.jsonl             0.8 MB
```

`train_strict_v1_negatives.jsonl` (7,586 pairs, keyword labels included) and
`train_full_v1_negatives.jsonl` are the ablations, not the default.

**Upload the fresh files.** If a previous Colab runtime is still alive it will
have the old ones on disk under the same names and the upload prompt will be
skipped silently — Runtime → Disconnect and delete runtime first.

In [ ]:
!pip -q install -U torchao
!pip -q install "sentence-transformers>=3.0" peft accelerate

# Colab's preinstalled torchao is older than peft's minimum, and peft checks it
# even though this notebook never uses torchao. If this is the first cell run
# in a fresh runtime the upgrade above is enough. If the LoRA cell still raises
# `ImportError: Found an incompatible version of torchao`, the old version was
# already imported into memory before the upgrade landed -- Runtime > Restart
# session, then Run all again (a plain re-run of later cells will not pick up
# the upgrade without a restart).

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# Must be set before torch allocates anything. Reduces allocator fragmentation
# under LoRA + gradient checkpointing's irregular allocation pattern; harmless
# if it turns out not to matter.

import gzip, json, time, collections, random
import numpy as np, torch

REQUIRED = ["train_anchor_v1_negatives.jsonl", "dev_anchor_v1.jsonl", "probe_questions.jsonl"]
def have_corpus(): return os.path.exists("corpus_v1.jsonl") or os.path.exists("corpus_v1.jsonl.gz")

missing = [f for f in REQUIRED if not os.path.exists(f)]
if missing or not have_corpus():
    try:
        from google.colab import files; files.upload()
    except Exception:
        pass

assert have_corpus(), "missing corpus_v1.jsonl(.gz)"
for f in REQUIRED:
    assert os.path.exists(f), f"missing {f}"

def read_jsonl(path):
    op = gzip.open if path.endswith(".gz") else open
    rows = []
    with op(path, "rt", encoding="utf-8") as fh:
        for ln, line in enumerate(fh, 1):
            line = line.strip()
            if not line: continue
            try: rows.append(json.loads(line))
            except Exception as e:
                raise SystemExit(f"{path} line {ln} corrupt ({e}) - upload truncated, re-upload the .gz")
    return rows

CORPUS_PATH = "corpus_v1.jsonl.gz" if os.path.exists("corpus_v1.jsonl.gz") else "corpus_v1.jsonl"
assert torch.cuda.is_available(), "no GPU - Runtime > Change runtime type > T4 GPU"
DEV = "cuda"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
chunks = read_jsonl(CORPUS_PATH)
train  = read_jsonl("train_anchor_v1_negatives.jsonl")
dev    = read_jsonl("dev_anchor_v1.jsonl")
probes = read_jsonl("probe_questions.jsonl")

assert len(chunks) == 39484, f"expected 39484 chunks, got {len(chunks)}"
assert len(probes) == 552,   f"expected 552 probe questions, got {len(probes)}"

# Document template: Act title(s) + provision title + body.
#
# Attempt 3 used `provision_title_bn + text_bn`, the same template as the
# zero-shot control, so that the comparison isolated the weights. That worked --
# R@10 0.324 -> 0.350, significant -- but it also means the two gains measured so
# far have never been combined:
#
#   act titles alone, no training : R@1 0.132, R@10 0.346
#   fine-tuning alone, old template: R@1 0.109, R@10 0.350
#
# They attack different failures. The Act title gives the encoder the name of the
# statute a section belongs to, which a citizen question often names and the
# section body never does. Fine-tuning moves citizen phrasing toward statute
# phrasing. Nothing says they overlap, so this run stacks them.
#
# The template is applied to the training texts as well as the index, because a
# model trained on bare sections and then asked to search title-prefixed ones is
# being tested on a distribution it never saw.
def document(c):
    bits = []
    for key in ("act_title_bn", "act_title_en"):
        v = (c.get(key) or "").strip()
        if v and v not in bits:
            bits.append(v)
    t = (c.get("provision_title_bn") or "").strip()
    if t:
        bits.append(t)
    bits.append(c["text_bn"])
    return " ".join(b for b in bits if b).strip()

text_of = {c["chunk_id"]: document(c) for c in chunks}
cid     = [c["chunk_id"] for c in chunks]
cprov   = [c["provision_id"] for c in chunks]

# Leakage assertion, restated here rather than trusted from upstream. A training
# question that appears in the probe set invalidates every number below.
probe_q = {p["question_bn"] for p in probes}
leaked  = {r["question_bn"] for r in train} & probe_q
assert not leaked, f"{len(leaked)} training questions are also probe questions"

train_pos_provs = {r["gold_provision_ids"][0] for r in train}
gold_provs = {p for r in probes for p in r["gold_provision_ids"]}
print(f"train {len(train)} pairs / {len({r['question_bn'] for r in train})} distinct questions")
print(f"dev   {len(dev)} pairs / {len({r['topic_id'] for r in dev})} held-out topics")
print(f"train positives that are gold provisions: {len(train_pos_provs & gold_provs)} (expect 0)")

# Label purity. The `anchor` variant keeps only positives named by hand in
# configs/synth_anchors.yaml. The keyword-matched labels it drops were measured
# on 2026-09-04 to be roughly 80% wrong at section level -- "maintenance"
# (a wife's খোরপোশ) matching the Chartered Accountants Ordinance's "Maintenance
# of branch offices", and so on. Under MNRL a wrong positive does not merely
# fail to teach; it pulls an unrelated provision toward citizen phrasing.
sources = collections.Counter(r.get("label_source") for r in train)
assert set(sources) == {"anchor"}, f"expected hand-anchored labels only, found {dict(sources)}"
print(f"label sources: {dict(sources)} over "
      f"{len(train_pos_provs)} distinct positive provisions")

## Training examples

`MultipleNegativesRankingLoss` treats every other item in the batch as a
negative, and additionally uses the explicit hard negatives when the example
carries them. That matters here: in-batch negatives on a corpus of statutes are
mostly trivially wrong (a fisheries regulation is not a plausible answer to a
divorce question), so without the mined ranks-5-30 negatives the loss collapses
early and the model learns almost nothing. The known-traps table calls this out
as "training loss goes to zero immediately".

**One row per pair, not one per negative.** The first version of this notebook
exploded each pair into up to 4 separate triplet rows -- same anchor, same
positive, different negative -- so the identical (question, answer) pair
appeared 4 times in the training data. At `batch_size=8` that made
same-batch collisions routine, and combined with a since-fixed bug in negative
mining (7.7% of mined negatives were actually a *different* pair's correct
answer -- see `scripts/17_mine_negatives.py`), that run collapsed the embedding
space: fine-tuned corpus vectors ended up at **mean cosine 0.16** against their
zero-shot counterparts on the same chunks -- not "shifted by training", nearly
unrelated. Recorded as a failed run in `DECISIONS.md` (2026-09-03), not as a
finding about fine-tuning.

Packing every pair's negatives into a single row of columns `anchor, positive,
negative_1, negative_2, ...` is what `MultipleNegativesRankingLoss` is meant to
consume: each pair appears exactly once, and its hard negatives contribute to
the loss for that one anchor without duplicating the anchor/positive across
rows.

Removing the duplication is necessary but not sufficient. Two rows can still
collide inside a batch without any duplication at all -- same question with a
different one of its anchored positives, or two different questions sharing an
anchored positive -- and MNRL reads both as "this correct answer is a negative".
The next cells measure that rate and batch with `NO_DUPLICATES` to remove it.

In [ ]:
from sentence_transformers import SentenceTransformer, losses

N_NEG = 3
# n_neg=3 (not the 8 the mined negatives allow): each row encodes 2 + n_neg
# texts per forward pass, so this is the cheapest lever for T4 memory that does
# not touch batch size -- batch size is what fixes attempt 1's collapse (more
# in-batch negative diversity), so it is the last thing to shrink, not the
# first. If OOM still happens after gradient checkpointing, drop this to 2
# before touching BATCH.

def to_columns(rows, n_neg=N_NEG):
    """Rows -> {anchor, positive, negative_1..n}. One row per pair, no duplication.

    MultipleNegativesRankingLoss reads column 1 as the anchor, column 2 as the
    positive, and every remaining column as an explicit negative for that anchor,
    on top of the in-batch negatives every other row contributes.

    Every row must carry the same number of columns, so a pair with fewer than
    n_neg mined negatives is dropped rather than padded. `17_mine_negatives.py`
    fills every pair to 8 whenever the rank window allows, so this should drop
    ~0 rows; if it drops many, that is worth noticing, not silently absorbing.
    """
    cols = {"anchor": [], "positive": [], **{f"negative_{i+1}": [] for i in range(n_neg)}}
    dropped_short = dropped_missing = 0
    for r in rows:
        pos = text_of.get(r["gold_chunk_ids"][0])
        if not pos:
            dropped_missing += 1
            continue
        negs = [text_of[n] for n in r.get("negative_chunk_ids", []) if n in text_of]
        if len(negs) < n_neg:
            dropped_short += 1
            continue
        cols["anchor"].append(r["question_bn"])
        cols["positive"].append(pos)
        for i in range(n_neg):
            cols[f"negative_{i+1}"].append(negs[i])
    if dropped_missing or dropped_short:
        print(f"  dropped {dropped_missing} pairs with an unresolvable positive, "
              f"{dropped_short} pairs with fewer than {n_neg} negatives")
    return cols


from datasets import Dataset

train_cols = to_columns(train)
train_ds = Dataset.from_dict(train_cols)
print(f"{len(train_ds)} training rows from {len(train)} pairs (1:1, no duplication), "
      f"columns {train_ds.column_names}")
print(f"  distinct anchors {len(set(train_cols['anchor']))}, "
      f"distinct positives {len(set(train_cols['positive']))}")


def collision_report(cols, batches, label):
    """Count same-anchor / same-positive pairs inside a given batching.

    Both are silent corruptions of MultipleNegativesRankingLoss, because the loss
    treats every other row's positive in the batch as a negative for this row:

    - **Same anchor, different positive.** Each synthetic question carries a
      median of 6 hand-anchored positives (real questions average 1.85 gold
      provisions, so multi-relevance is genuine, not a bug). One row per pair
      means a question appears ~6 times in the data; when two land in one batch,
      the loss is told that question Q's correct section A is a negative for
      question Q.
    - **Different anchor, same positive.** Two questions from one topic often
      share an anchored provision, so row B's positive text is identical to row
      A's, and the loss pushes query B away from its own answer.

    Measured on this exact dataset at batch 16, shuffled: 65 same-anchor and 190
    same-positive collisions across the 333 batches of one epoch. Neither raises
    an error and neither is visible in the loss curve. Same family as attempt 1's
    collapse, where each pair was exploded into 4 duplicate rows -- that fix
    removed the duplication, this one removes the collisions that survive
    without any duplication at all.
    """
    same_anchor = same_positive = 0
    for b in batches:
        anchors = [cols["anchor"][i] for i in b]
        positives = [cols["positive"][i] for i in b]
        same_anchor += len(anchors) - len(set(anchors))
        same_positive += len(positives) - len(set(positives))
    print(f"  {label:24s} {len(batches):4d} batches  same-anchor {same_anchor:4d}  "
          f"same-positive {same_positive:4d}")


In [ ]:
CHECKPOINT = "BAAI/bge-m3"
MAX_SEQ    = 512      # identical to the zero-shot index
BATCH      = 16       # T4 16GB with LoRA + gradient checkpointing. A bigger
                       # batch gives MultipleNegativesRankingLoss more, and more
                       # varied, in-batch negatives per step. Those in-batch
                       # negatives are the force that keeps the embedding space
                       # spread out, working against the pull-toward-positive
                       # that concentrated the query space in attempt 2 -- so
                       # this is the LAST thing to shrink for an OOM, not the
                       # first. Gradient checkpointing below is the right lever.
EPOCHS     = 1        # was 2. Attempt 2's query concentration drifted +0.12
                       # over two epochs; the drift is progressive, so halving
                       # the exposure is the cheapest brake.
LR         = 1e-5     # was 2e-5, same reasoning. Between them these two changes
                       # roughly quarter the total parameter movement.

model = SentenceTransformer(CHECKPOINT, device=DEV)
model.max_seq_length = MAX_SEQ

# LoRA rather than full fine-tuning: BGE-m3 is 568M params and a full backward
# pass at seq-512 does not fit a T4 alongside the optimiser state. LoRA on the
# attention projections is also the safer choice for a dataset this size - a
# full fine-tune on this little data would move the encoder far enough to lose
# the multilingual alignment that makes the English Acts reachable at all, which
# is the one capability this project cannot afford to damage.
#
# target_modules is ["query", "value"], narrowed from attempt 1's
# ["query", "key", "value", "dense"]. "dense" is a substring match against
# HuggingFace's XLM-RoBERTa naming, so it hit THREE separate Linear layers per
# transformer block (attention.output.dense, intermediate.dense, output.dense)
# on top of q/k/v -- far more adapter capacity than this data can fit without
# drifting the encoder. q+v is the conservative standard recipe for retrieval
# fine-tuning; putting k and the dense layers back is a deliberate ablation to
# run only once q+v is shown to be safe.
from peft import LoraConfig, get_peft_model

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules=["query", "value"],
)
backbone = model[0].auto_model

# Keep the wrapper in its OWN variable and call print_trainable_parameters()
# on THAT, rather than re-reading it back through model[0].auto_model. Reading
# it back immediately after assignment can resolve to the original unwrapped
# HF model instead of the PEFT wrapper (observed: `AttributeError:
# 'XLMRobertaModel' object has no attribute 'print_trainable_parameters'`) --
# apparently an nn.Module attribute-resolution quirk between sentence-
# transformers' Transformer module and PEFT's wrapper. Never diagnosed further
# because it does not need to be: holding the reference we already have
# sidesteps the ambiguity entirely.
peft_backbone = get_peft_model(backbone, lora)
peft_backbone.print_trainable_parameters()

# Gradient checkpointing: trades ~20-30% training speed for a large cut in
# activation memory, which is what a T4's 15GB actually runs out of here (each
# training example encodes 2 + n_neg texts, so a batch of 16 is ~80 forward
# passes' worth of activations at seq-512 on a 568M-param backbone).
# `enable_input_require_grads()` is required alongside it: with the backbone
# frozen (only the LoRA adapters have requires_grad=True), checkpointing can
# otherwise produce a graph with no tensor requiring grad at the recomputation
# boundary, and silently drop gradients rather than erroring.
peft_backbone.gradient_checkpointing_enable()
peft_backbone.enable_input_require_grads()

model[0].auto_model = peft_backbone

In [ ]:
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.training_args import BatchSamplers
try:                                      # moved in sentence-transformers 6.x
    from sentence_transformers.base.sampler import NoDuplicatesBatchSampler
except ImportError:
    from sentence_transformers.sampler import NoDuplicatesBatchSampler

loss = losses.MultipleNegativesRankingLoss(model)

# Why the Trainer API and not model.fit(). `fit()` is deprecated in
# sentence-transformers v3+, and -- more importantly -- it DISCARDS a custom
# batch sampler: it rebuilds a Dataset from whatever DataLoader you hand it and
# then picks the batch sampler itself, recognising only its own legacy
# `NoDuplicatesDataLoader` class. Passing a hand-written sampler to `fit()`
# looks like it works and silently trains with plain random batching. (The
# legacy `NoDuplicatesDataLoader` is also the wrong tool: it cycles the dataset
# looking for a conflict-free batch and never terminates when the duplicates are
# dense relative to the batch size -- verified by hanging it locally on a toy
# set with 5 distinct anchors at batch 8.) The Trainer's own
# NoDuplicatesBatchSampler defers conflicting rows through a linked list and
# terminates cleanly, so that is what runs.
#
# NO_DUPLICATES dedupes on ALL text columns, not just anchor and positive, so it
# also prevents one row's mined negative from being another row's positive in
# the same batch -- a third false-negative path, free.

# The baseline must be a SHUFFLED batching, which is what a plain trainer does.
# Using file order instead would flatter the fix: the file is grouped by
# question, so contiguous batches collide constantly (4,242 same-anchor
# collisions measured that way) and the comparison would be against a strawman.
_perm = list(range(len(train_ds)))
random.Random(13).shuffle(_perm)
naive = [_perm[i:i + BATCH] for i in range(0, len(_perm) - BATCH + 1, BATCH)]
dedup = list(NoDuplicatesBatchSampler(train_ds, batch_size=BATCH, drop_last=True))
print("in-batch contradictions:")
collision_report(train_cols, naive, "plain random batching")
collision_report(train_cols, dedup, "NO_DUPLICATES sampler")
assert sum(len({train_cols["anchor"][i] for i in b}) != len(b) for b in dedup) == 0,     "sampler did not remove same-anchor collisions"
print(f"{len(dedup)} steps/epoch at batch {BATCH}")

args = SentenceTransformerTrainingArguments(
    output_dir="ft_out",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    learning_rate=LR,
    warmup_ratio=0.1,
    fp16=True,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    logging_steps=25,
    save_strategy="no",
    report_to=[],
    seed=13,
)

t = time.time()
trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=train_ds, loss=loss)
trainer.train()
print(f"trained in {(time.time()-t)/60:.1f} min")

# If this OOMs, escalate in this order (each keeps more of the collapse fix than
# the next):
#   1. N_NEG=2 in the previous cell.
#   2. gradient_accumulation_steps=2 with BATCH=8 -- note this does NOT restore
#      the in-batch negative count, it only fakes the optimiser step size.
#   3. BATCH=8, last resort: this is the exact lever that let attempt 1's
#      cross-batch collisions dominate.


In [ ]:
## Canary: did training move the embedding space, or wreck it?
#
# Two failure modes, both observed for real, both cheap to catch here instead
# of after a 10-minute full-corpus embed and a full eval.
#
# (1) COLLAPSE (attempt 1): fine-tuned corpus embeddings ended up at mean
#     cosine 0.16 against zero-shot on the SAME chunks -- nearly unrelated,
#     R@10 fell 0.324 -> 0.060. Caught by `canary_cosine`.
#
# (2) QUERY-SPACE CONCENTRATION / HUBNESS (attempt 2): no collapse
#     (cosine 0.83, healthy) but the *queries* bunched together -- mean cosine
#     to the query centroid rose 0.699 -> 0.822 -- and whichever documents sat
#     nearest that centroid became universal top-1 attractors. Two chunks took
#     43 and 44 of the 333 English top-1 slots; distinct top-1 documents fell
#     from 168 to 144. Overall recall went flat and English R@1/R@5 got
#     significantly WORSE, while deep recall (R@50/R@100) improved on one
#     slice. It is not memorisation -- the worst hub had zero training pairs as
#     a positive. Caught by `query_concentration`.
#
#     Note what the cause is NOT. The first guess was that the training
#     questions were intrinsically homogeneous. Measured against this same base
#     encoder they were not: mean cosine to the question centroid was 0.6983 for
#     the 413 templates against 0.6990 for the 552 real probe questions, and
#     mean pairwise cosine 0.4863 vs 0.4875 (scripts/22_question_diversity.py,
#     2026-09-04). The training set entered no narrower than the test set. The
#     concentration was therefore INDUCED by the optimisation -- with few
#     distinct anchors, one epoch's worth of MNRL pull-toward-positive
#     outweighing the spreading force of only 15 in-batch negatives per step --
#     which is what EPOCHS, LR and BATCH below are set against.
#
#     A separate, real defect did show up in the same audit: length. The
#     templates ran median 11 words (sd 3.4) against 74 (sd 56.3) for the real
#     letters. That is a train/test distribution gap on its own and is fixed
#     upstream by the narrative scaffolding in configs/synth_narrative.yaml
#     (now median 57, sd 30.3), not by anything in this notebook.
#
# Healthy run: canary_cosine 0.7-0.98, and query_concentration not much above
# the base model's own value on the same questions.
import random as _random

sample_ids = _random.Random(0).sample(cid, 200)
sample_docs = [text_of[c] for c in sample_ids]
probe_texts = [p["question_bn"] for p in probes]

base_model = SentenceTransformer(CHECKPOINT, device=DEV)
base_model.max_seq_length = MAX_SEQ
base_vecs  = base_model.encode(sample_docs, batch_size=32, normalize_embeddings=True,
                               convert_to_numpy=True)
base_probe = base_model.encode(probe_texts, batch_size=32, normalize_embeddings=True,
                               convert_to_numpy=True)
del base_model; torch.cuda.empty_cache()

ft_vecs  = model.encode(sample_docs, batch_size=32, normalize_embeddings=True,
                        convert_to_numpy=True)
ft_probe = model.encode(probe_texts, batch_size=32, normalize_embeddings=True,
                        convert_to_numpy=True)

canary_cosine = float((base_vecs * ft_vecs).sum(axis=1).mean())
print(f"canary: mean cosine(base, fine-tuned) on {len(sample_ids)} sampled chunks = {canary_cosine:.4f}")

def concentration(Q):
    """Mean cosine of each query to the query centroid. Higher = queries bunched."""
    Qn = Q / np.linalg.norm(Q, axis=1, keepdims=True)
    cen = Qn.mean(0); cen /= np.linalg.norm(cen)
    return float((Qn @ cen).mean())

base_concentration  = concentration(base_probe)
query_concentration = concentration(ft_probe)
print(f"query concentration (mean cosine to query centroid): "
      f"base {base_concentration:.4f} -> fine-tuned {query_concentration:.4f} "
      f"(delta {query_concentration - base_concentration:+.4f})")

CANARY_FLOOR = 0.5
CONCENTRATION_DELTA_CEILING = 0.06   # attempt 2 drifted +0.12 and lost top-of-list precision

if canary_cosine < CANARY_FLOOR:
    raise SystemExit(
        f"canary failed: {canary_cosine:.4f} < {CANARY_FLOOR} -- this looks like the "
        f"attempt-1 embedding-space collapse, not a real fine-tune. Do not spend GPU "
        f"time on the full corpus embed or the probe eval below. Re-check the training "
        f"loss curve, confirm scripts/17_mine_negatives.py's global-exclusion fix and "
        f"to_examples()'s fixed-length assertion both ran, and re-train before continuing."
    )

if query_concentration - base_concentration > CONCENTRATION_DELTA_CEILING:
    print(
        f"\n*** WARNING: query concentration rose {query_concentration - base_concentration:+.4f} "
        f"(ceiling {CONCENTRATION_DELTA_CEILING}).\n"
        f"*** This is attempt 2's failure mode: queries bunch toward one direction, hub\n"
        f"*** documents near that direction take top-1 for unrelated questions, and R@1/R@5\n"
        f"*** degrade even when deep recall improves. The eval below still runs -- the\n"
        f"*** numbers are real and worth recording -- but expect flat-to-worse top-of-list\n"
        f"*** results. The fix is STRUCTURALLY more varied training questions (varied\n"
        f"*** sentence frames, lengths, narrative shapes -- not merely more topics), plus\n"
        f"*** a lower LR and/or fewer epochs so the encoder drifts less far.\n"
    )
else:
    print("canary passed -- proceeding to the full corpus embed.")

## Dev check on held-out topics

Dev is 6 topics the model never saw - land acquisition, maternity leave, road
accident, wills/probate, workplace injury, wrongful termination. If the model
only memorised the 30 training topics, dev recall stays flat while training loss
falls. That is the cheap early warning, and it costs one matmul.

In [ ]:
def embed_corpus(m, batch=64):
    docs = [text_of[c] for c in cid]
    return m.encode(docs, batch_size=batch, normalize_embeddings=True,
                    convert_to_numpy=True, show_progress_bar=True).astype(np.float16)

def recall_at(m, rows, ks=(1, 5, 10, 20, 50, 100), corpus_emb=None):
    C = corpus_emb if corpus_emb is not None else embed_corpus(m)
    Q = m.encode([r["question_bn"] for r in rows], batch_size=64,
                 normalize_embeddings=True, convert_to_numpy=True)
    Cg = torch.tensor(C, device=DEV, dtype=torch.float16)
    Qg = torch.tensor(Q, device=DEV, dtype=torch.float16)
    hits = collections.Counter()
    depth = max(ks)
    for i in range(0, len(rows), 64):
        idx = torch.topk(Qg[i:i+64] @ Cg.T, depth, dim=1).indices.cpu().numpy()
        for j, order in enumerate(idx):
            gold = set(rows[i+j]["gold_provision_ids"])
            rank = next((r for r, ci in enumerate(order, 1) if cprov[ci] in gold), None)
            for k in ks:
                if rank and rank <= k: hits[k] += 1
    del Cg, Qg; torch.cuda.empty_cache()
    return {f"R@{k}": round(hits[k] / len(rows), 4) for k in ks}

t = time.time()
C_ft = embed_corpus(model)
print(f"corpus embedded {C_ft.shape} in {(time.time()-t)/60:.1f} min")

# Dedupe dev to distinct questions so a topic with many anchors does not dominate.
dev_unique = list({r["question_bn"]: r for r in dev}.values())
print(f"dev (held-out topics, n={len(dev_unique)}):", recall_at(model, dev_unique, corpus_emb=C_ft))

## The real evaluation: 552 gold questions, full corpus

Same 552 questions, same 39,484 chunks, same document construction as the
zero-shot run. The artifacts below are written in the layout
`scripts/13_eval_dense_index.py` expects, so the fine-tuned numbers are produced
by the identical scoring code that produced the control - including `per_query`,
which the paired bootstrap needs.

In [ ]:
Q_probe = model.encode([p["question_bn"] for p in probes], batch_size=64,
                       normalize_embeddings=True, convert_to_numpy=True).astype(np.float16)

# Save the adapter this time. Attempt 3's weights were lost when the runtime
# ended, so testing any new document template meant retraining from scratch --
# 30 minutes to answer a question the saved adapter would have answered in five.
peft_backbone.save_pretrained("lora_adapter")
import shutil
shutil.make_archive("lora_adapter", "zip", "lora_adapter")
print("saved lora_adapter.zip -- download this alongside the index files")

np.save("embeddings.npy", C_ft)
json.dump(cid, open("chunk_ids.json", "w"))
np.save("probe_query.npy", Q_probe)

manifest = {
    "checkpoint": CHECKPOINT, "max_seq_length": MAX_SEQ,
    "use_title": True, "query_prefix": "", "passage_prefix": "",
    "dim": int(C_ft.shape[1]), "dtype": "float16", "normalized": True,
    "n_chunks": int(C_ft.shape[0]), "corpus_file": "data/processed/corpus_v1.jsonl",
    "fine_tuned": True,
    "train_file": "data/processed/train_anchor_v1_negatives.jsonl",
    "train_variant": "anchor",
    "train_pairs": len(train), "train_rows": len(train_ds),
    "train_label_source": "anchor",
    "train_positive_provisions": len(train_pos_provs),
    "n_neg_per_row": N_NEG,
    "batch_sampler": "NO_DUPLICATES",
    "steps_per_epoch": len(dedup),
    "lora": {"r": 16, "alpha": 32, "dropout": 0.05, "target_modules": ["query", "value"]},
    "epochs": EPOCHS, "batch_size": BATCH, "lr": LR,
    "canary_cosine_vs_base": round(canary_cosine, 4),
    "query_concentration_base": round(base_concentration, 4),
    "query_concentration_finetuned": round(query_concentration, 4),
    "built_by": "notebooks/colab_bge_m3_finetune.ipynb",
    "document_template": "act_title_bn + act_title_en + provision_title_bn + text_bn",
    "use_act_title": True,
    "notes": "Fine-tuned arm, attempt 3. Compare against models/index_bge_m3_zeroshot_v1 "
             "via scripts/13. Guards against three measured failure modes: embedding "
             "collapse (attempt 1) and query-space hubness (attempt 2), both in "
             "DECISIONS.md 2026-09-03; and, new here, wrong labels plus in-batch "
             "contradictions -- the keyword-matched positives were ~80% wrong at section "
             "level and are dropped, and NO_DUPLICATES batching removes the same-anchor "
             "and same-positive collisions that plain batching leaves in.",
}
json.dump(manifest, open("manifest.json", "w"), indent=2)

# Printed here only as an immediate sanity check. The citable numbers come from
# scripts/13_eval_dense_index.py run against these files back in the repo.
print("probe (552, full corpus):", recall_at(model, probes, corpus_emb=C_ft))
print("zero-shot control       :", {"R@1": 0.096, "R@5": 0.239, "R@10": 0.324,
                                    "R@20": 0.413, "R@50": 0.534, "R@100": 0.627})
print("attempt 1 (collapsed)   :", {"R@1": 0.009, "R@5": 0.034, "R@10": 0.060,
                                    "R@20": 0.111, "R@50": 0.219, "R@100": 0.313})
print("attempt 2 (hubness)     :", {"R@1": 0.083, "R@5": 0.219, "R@10": 0.330,
                                    "R@20": 0.415, "R@50": 0.556, "R@100": 0.658})

In [ ]:
model.save("dhara_bge_m3_lora")
!zip -qr dhara_bge_m3_lora.zip dhara_bge_m3_lora

try:
    from google.colab import files
    for f in ["embeddings.npy", "chunk_ids.json", "probe_query.npy",
              "manifest.json", "dhara_bge_m3_lora.zip"]:
        files.download(f)
except Exception:
    print("not on Colab - download these from the file panel:")
    print("  embeddings.npy chunk_ids.json probe_query.npy manifest.json dhara_bge_m3_lora.zip")

## Back in the repo

```bash
mkdir -p models/index_bge_m3_finetuned_v3
mv embeddings.npy chunk_ids.json probe_query.npy manifest.json models/index_bge_m3_finetuned_v3/

python scripts/13_eval_dense_index.py \
    --index models/index_bge_m3_finetuned_v3 --run-id bge_m3_finetuned_strict_v3

python scripts/18_compare_runs.py \
    --a results/runs/bge_m3_finetuned_strict_v3.json \
    --b results/runs/bge_m3_zeroshot.json

# And the instrument that explained attempt 2's flat result, which aggregate
# recall could not see. Run it even if the recall numbers look fine.
python scripts/20_hubness_audit.py \
    --a models/index_bge_m3_finetuned_v3 --a-id bge_m3_finetuned_strict_v3 \
    --b models/index_bge_m3_zeroshot_v1  --b-id bge_m3_zeroshot
```

`18_compare_runs.py` reports the paired-bootstrap 95% CI on the difference at
every cutoff and for every language slice. At n=552 the CI on a recall
difference is roughly ±4 points, so a 2-point gain is not a gain.

Each attempt writes its own run-id and its own index directory — nothing
overwrites anything. `bge_m3_finetuned_strict` (attempt 1, collapsed) and
`_v2` (attempt 2, hubness) both stay on disk and in DECISIONS.md as recorded
failures. Three honestly-reported attempts with a diagnosed cause each is a
stronger result than one lucky number.

### Reading attempt 3

- **Canary cosine 0.7-0.98** and **concentration delta under +0.06** — the run
  is structurally healthy, so the recall numbers mean what they say.
- **Concentration delta above +0.06** — the hubness mode returned despite the
  narrative rewrite and the lower LR. That would be genuinely informative: it
  would point at the contrastive objective on a small corpus rather than at the
  training data, and the next lever is a larger batch or an explicit uniformity
  term, not more question variety.
- **Flat again with a healthy canary** — the honest conclusion is that
  fine-tuning a bi-encoder on this much synthetic data does not move this task,
  and the reranker (`notebooks/colab_acttitle_and_rerank.ipynb`) is where the
  remaining headroom actually is: 30.3% of questions have their gold provision
  sitting in ranks 11-100 right now.